# Лабораторная работа №4 «Численное интегрирование»
Выполнил: Макаров Станислав Алексеевич, ИВТб-2301<br>
Вариант 14

In [3]:
from IPython.display import display, Markdown, HTML

def md(s):
    display(Markdown(s))

def tb(headers, rows):
    header_html = ''.join(f'<th>{header}</th>' for header in headers)
    body_html = ''.join(
        '<tr>' + ''.join(f'<td>{cell}</td>' for cell in row) + '</tr>'
        for row in rows
    )
    display(HTML(
        f'''
        <style>
          .table th,
          .table td {{
            border: 1px solid #000;
            min-width: 4rem;
            padding: 0.35rem 0.8rem;
            text-align: center;
          }}
        </style>
        <table class="table">
          <thead><tr>{header_html}</tr></thead>
          <tbody>{body_html}</tbody>
        </table>
        '''
    ))

## Задание 1. Формула трапеций

Вычислить определенный интеграл с точностью $\varepsilon=0.0001$:
$$I = \int_{1.4}^{2.2} \frac{1}{\sqrt{3*x^2+1}}dx.$$

In [4]:
import math

def f(x):
    return 1 / math.sqrt(3 * x**2 + 1)

Найдем вторую производную:
$$f''(x) = \frac{3(6x^2-1)}{(3x^2+1)^\frac{5}{2}}.$$

In [5]:
def ddf(x):
    return (3 * (6 * x**2 - 1)) / ((3 * x**2 + 1)**2.5)

Для нахождения числа $n$ используем формулу остаточного члена:
$$|R_T| \le \frac{(b-a)^3}{12n^2} \max_{[a,b]} |f''(x)|.$$

In [6]:
def rt(a, b, x, eps = 1e-4):
    return math.sqrt(((b - a) ** 3 * abs(ddf(x))) / (12 * eps))

Используем формулу трапеций для вычисления определенного интеграла:
$$\int_{a}^{b} f(x)dx \approx \frac{b-a}{n}(f(a) + f(b))$$

In [7]:
def trapezoid(a, b, n):
    h = (b - a) / n
    total = (f(a) + f(b)) / 2

    for k in range(1, n):
        total += f(a + k * h)

    return h * total

На $[1.4, 2.2]$ |f''(x)| достигает максимума на левой границе $x=1.4$ &rArr; $max|f''(x)|=|f''(1.4)|$

In [8]:
a, b, = 1.4, 2.2
n = math.ceil(rt(a, b, a))
I = trapezoid(a, b, n)

md(f"Значение определенного интегра $I = {I}$")

Значение определенного интегра $I = 0.24758711007019452$

## Задание 2. Формула Симпсона

Вычислить определенный интеграл с точностью $\varepsilon=0.0001$:
$$I = \int_{1.4}^{2.2} \frac{lg(x^2+2)}{x+1}dx.$$

In [9]:
def f(x):
    return math.log(x**2 + 2) / (x + 1)

Используем формулу Симпсона для вычисления определенного интеграла:
$$\int_{a}^{b}f(x)dx \approx \frac{b-a}{6}[f(a)+4f(\frac{a+b}{2}+f(b)]$$

In [10]:
def simpson(a, b, n):
    h = (b - a) / n
    total = f(a) + f(b)

    for k in range(1, n):
        total += (4 if k % 2 else 2) * f(a + k * h)

    return (h / 3) * total

В качестве начального шага возьмем число, близкое к $e^\frac{1}{4}$. Для приближенной оценки погрешности применим принцип Рунге:
$$|R| \approx \frac{|I_h - I_{2h}|}{15}.$$

In [11]:
n = 2
I_h = simpson(a, b, n)
I_2h = simpson(a, b, n * 2)
R = abs(I_h - I_2h) / 15

md(f"$I_h = {I_h}$")
md(f"$I_h = {I_2h}$")
md(f"$R = {R}$")

$I_h = 0.472063966094734$

$I_h = 0.4720498723038091$

$R = 9.395860616582633e-07$

## Задание 3. Квадратурная формула Гаусса

Вычислить определенный интеграл:
$$I = \int_{2.4}^{3.2} \frac{x^2}{\sqrt{x+2}}dx.$$

In [12]:
import math

def f(x):
    return x**2 / math.sqrt(x + 2)

Используем квадратурную формулу Гаусса для вычисления определенного интеграла:
$$\int_{a}^{b}f(x)dx \approx \frac{b-a}{2}\sum_{i=1}^{n}w_if(\frac{b-a}{2}x_i+\frac{a+b}{2})$$

In [13]:
def gauss(a, b, nodes, weights):
    mid = (a + b) * 0.5
    half = (b - a) * 0.5
    return half * sum(w * f(mid + half * t) for t, w in zip(nodes, weights))

In [14]:
a, b, = 2.4, 3.2
nodes4 = [-0.86114, -0.33998, 0.33998, 0.86114]
weights4 = [0.34785, 0.65215, 0.65215, 0.34785]
nodes7 = [-0.949107912, -0.741531186, -0.405845151, 0.0, 0.405845151, 0.741531186, 0.949107912]
weights7 = [0.129484966, 0.279705391, 0.381830051, 0.417959184, 0.381830051, 0.279705391, 0.129484966]

I_4 = gauss(a, b, nodes4, weights4)
I_7 = gauss(a, b, nodes7, weights7)

md(f"$I_4 = {I_4}$")
md(f"$I_7 = {I_7}$")
md(f"$Err = {abs(I_7 - I_4)}$")

$I_4 = 2.8733710595862836$

$I_7 = 2.8733711004472333$

$Err = 4.086094973487775e-08$

## Задание 4. Решение ОДУ по формуле 2-го порядка точности

Решить обыкновенное дифференциальное уравнение:
$$y'=2*x-1+y^2, \qquad a=\frac{1}{2}, \qquad y(0)=1, \qquad h=0.1, \qquad 0\le x \le 1.$$

In [15]:
def f(x, y):
    return 2 * x - 1 + y**2

Для решения уравнения используем формулу 2-го порядка точности:
$$\Delta y_k = \frac{h}{2}(f(x_k,y_k)+f(x_k+h,y_k+hf_k)), \qquad y_{k+1}=y_k+\Delta y_k.$$

In [16]:
def second_order(x, y, h, x_end):
    results = []
    results.append({
        'k': 0, 'x': x, 'y': y, 'f1': None,
        'x_next': None, 'y_pred': None, 'f2': None,
        'delta_y': None
    })

    while x < x_end - 1e-5:
        f1 = f(x, y)
        y_pred = y + h * f1
        x_next = x + h
        f2 = f(x_next, y_pred)
        delta_y = (h / 2) * (f1 + f2)
        y_next = y + delta_y

        results.append({
            'k': len(results),
            'x': x_next,
            'y': y_next,
            'f1': f1,
            'x_next': x_next,
            'y_pred': y_pred,
            'f2': f2,
            'delta_y': delta_y
        })

        x, y = x_next, y_next

    return results

In [17]:
results_h = second_order(0.0, 1.0, 0.2, 1.0)
results_half = second_order(0.0, 1.0, 0.2 / 2, 1.0)

y_half_at_grid = {0.0: results_half[0]['y']}
for res in results_half:
    for res_h in results_h:
        if abs(res['x'] - res_h['x']) < 1e-10:
            y_half_at_grid[res_h['x']] = res['y']
            break

for res_h in results_h:
    x = res_h['x']
    y_h = res_h['y']
    y_half = y_half_at_grid.get(x, None)
    
    if y_half is not None and res_h['k'] > 0:
        error_runge = abs(y_h - y_half) / 3.0
        res_h['error_runge'] = error_runge
        res_h['y_half'] = y_half
    else:
        res_h['error_runge'] = 0.0
        res_h['y_half'] = y_half if y_half is not None else 1.0

headers = [
    'k', 'x_k', 'y_k',
    'f(x_k, y_k)', 'x_k + h', 'y_k + h·f_k',
    'f(x_k + h, y_k + h·f_k)', 'Δy_k', 'y_точное', 'ε'
]

rows = []
for r in results_h:
    if r['k'] == 0:
        rows.append([
            '0',
            f"{r['x']:.4f}",
            f"{r['y']:.4f}",
            '',
            '',
            '',
            '',
            '',
            f"{r['y']:.4f}",
            '0'
        ])
    else:
        rows.append([
            str(r['k']),
            f"{r['x']:.4f}",
            f"{r['y']:.4f}",
            f"{r['f1']:.4f}",
            f"{r['x_next']:.4f}",
            f"{r['y_pred']:.4f}",
            f"{r['f2']:.4f}",
            f"{r['delta_y']:.4f}",
            f"{r['y_half']:.4f}",
            f"{r['error_runge']:.4f}"
        ])

tb(headers, rows)

k,x_k,y_k,"f(x_k, y_k)",x_k + h,y_k + h·f_k,"f(x_k + h, y_k + h·f_k)",Δy_k,y_точное,ε
0,0.0000,1.0000,,,,,,1.0000,0
1,0.2000,1.0400,0.0000,0.2000,1.0000,0.4000,0.0400,1.0443,0.0014
2,0.4000,1.1973,0.4816,0.4000,1.1363,1.0912,0.1573,1.2110,0.0046
3,0.6000,1.5491,1.2335,0.6000,1.4440,2.2851,0.3519,1.5868,0.0126
4,0.8000,2.2972,2.5998,0.8000,2.0691,4.8812,0.7481,2.4186,0.0404
5,1.0000,4.1909,5.8773,1.0000,3.4727,13.0597,1.8937,4.8306,0.2132
